# EduGenine AI — 맞춤형 AI 학습 도우미

가천대학교 2024-2학기 「AI 중심세상」 과목 팀프로젝트에서 본인이 작성한 코드를 정리한 개인 저장소의 노트북입니다. 팀 전체 결과물의 공식 저장소는 아닙니다. 여러 과목의 문제·해설·피드백을 제공하는 학습 도우미를 목표로 하며, 이 노트북은 이산수학 예제로 학습 흐름을 보여줍니다. 프로젝트 소개는 **README.md**, 구현 및 실험 안내는 **docs/IMPLEMENTATION.md**를 참고하세요.

**기본 실행:** 검토한 문제은행 10문항에서 문제를 선택하고, 실제 정답으로 채점하여 난이도·피드백·복습 목록을 갱신합니다. 이 모드는 생성형 AI를 사용하지 않습니다.

**선택 실험:** GPT-2 파인튜닝과 LSTM 학습을 별도 모듈로 구성했습니다. 검토를 마친 패키지와 로컬 모델이 있어야 켤 수 있습니다. 모델 성능을 검증했다는 의미는 아닙니다.

Python 3.10 이상을 사용하세요. 기본 설정으로 전체 실행해도 설치·다운로드·입력 대기·파일 저장이 발생하지 않습니다.

## 1. 프로젝트와 실행 옵션

이 노트북과 `edugenine.py`, `ai_experiments.py`, `data` 폴더를 함께 보관하세요. 프로젝트 폴더에서 노트북을 실행합니다. 다른 작업 폴더에서 실행한다면 `PROJECT_DIR`을 실제 프로젝트 경로로 지정하세요.

In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import asdict

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'edugenine.py').is_file():
    PROJECT_DIR = PROJECT_DIR / 'EduGenine-AI'
if not (PROJECT_DIR / 'edugenine.py').is_file():
    raise FileNotFoundError('PROJECT_DIR을 EduGenine-AI 폴더의 절대 경로로 지정하세요.')
PROJECT_DIR = PROJECT_DIR.resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from edugenine import (load_questions, LearningSession, run_interactive,
                      export_session, load_material, suggest_topics)

SEED = 42
INTERACTIVE = False
SAVE_REPORT = False
MATERIAL_PATH = None                 # 예: PROJECT_DIR / 'my_notes.txt'
REVIEWED_PDF_ENV = False              # PyPDF2와 의존성 검토를 마친 경우에만 변경
RUN_AI_EXPERIMENT = False
REVIEWED_AI_ENV = False               # 패키지·모델 검토를 대신하는 플래그가 아닙니다.
MODEL_DIR = PROJECT_DIR / 'models' / 'reviewed-gpt2'
print('프로젝트:', PROJECT_DIR.name)
print('기본 실행: Python 표준 라이브러리 기반 문제은행 데모')

## 2. 검토한 문제 데이터

10문항의 정답과 조건을 명시했습니다. `2n+3`은 모든 정수 n에 대해 홀수이고, 최단 경로에는 방향 간선 정보를 추가했습니다. 집합의 답안을 생략 없이 기입하고 피보나치 초기값도 명시했습니다. 유한한 항만으로 다음 항을 단정하지 않도록 제곱수 문제의 일반항을 명시했습니다.

이 데이터는 KorQuAD가 아닙니다. 이 저장소는 10문항 예제 데이터만 포함합니다. 난이도 1~5는 데모를 위한 수동 분류이며 실제 데이터에는 1~3단계 문제가 있습니다.

In [ ]:
questions = load_questions(PROJECT_DIR / 'data' / 'discrete_math.json')
print(f'검토한 문제: {len(questions)}개')
for q in questions:
    print(f'{q.id} | {q.topic} | 난이도 {q.level} | {q.question}')

## 3. 채점과 학습 정책

숫자는 소수·분수를 동등하게, 집합은 원소 순서에 무관하게, 경로는 이동 순서를 유지해 비교합니다. 예를 들어 `1/2`와 `0.5`, `{6, 4}`와 `{4, 6}`은 같은 답입니다. 자유서술형 의미 채점이나 부분점수는 지원하지 않으며, 텍스트는 데이터의 정답 또는 명시적 별칭과 일치해야 합니다.

정답이면 목표 난이도를 1단계 올리고 오답·시간 초과이면 1단계 내립니다. 다음 문제는 풀지 않은 문제 중 최근 오답 주제와 목표 난이도를 고려해 선택합니다. 같은 문제를 한 세션에서 반복 출제하지 않습니다. 3회 연속 오답일 때 복습 안내를 제공합니다.

In [ ]:
# 동작 확인용 합성 데모입니다. 실제 사용자 성적이나 AI 성능이 아닙니다.
demo_session = LearningSession(questions, level=2, seed=SEED)
for index in range(5):
    question = demo_session.next_question()
    demo_answer = '오답 예시' if index in (1, 2) else question.answer
    attempt = demo_session.submit(question, demo_answer, elapsed_seconds=20 + index * 5)
    print(f'{question.id}: {"정답" if attempt.correct else "오답"} / '
          f'목표 난이도 {attempt.level_before} → {attempt.level_after}')
    print(attempt.feedback)
print('합성 데모 요약:', json.dumps(demo_session.summary(), ensure_ascii=False, indent=2))

## 4. 학습 자료 입력과 주제 확인 (선택)

`MATERIAL_PATH`에 UTF-8 TXT/MD 파일을 지정하면 키워드로 문제은행의 주제를 제안합니다. 자료에 없는 내용을 추론해 주제를 확정하지 않습니다. PDF는 검토된 PyPDF2 환경에서 `REVIEWED_PDF_ENV=True`인 경우에만 텍스트를 추출합니다. 스캔 PDF의 OCR은 지원하지 않습니다.

주제 연결은 키워드 일치에 기반하며 자료 요약 기능은 제공하지 않습니다. 기본 모드는 자료를 바탕으로 새 문제를 생성하지 않습니다.

In [ ]:
material_text = ''
if MATERIAL_PATH is not None:
    material_text = load_material(MATERIAL_PATH, reviewed_pdf_environment=REVIEWED_PDF_ENV)
    print('문제은행과 연결 가능한 주제:', suggest_topics(material_text, questions))
else:
    print('자료 경로를 지정하면 주제 후보를 표시합니다.')

## 5. 직접 문제 풀기 (선택)

맨 위에서 `INTERACTIVE=True`로 바꾸고 실행하세요. `TOPIC=None`은 전체 주제, 예를 들어 `'집합'`은 집합만 선택합니다. `exit`로 중단합니다.

3분 제한은 **제출 후** 판정합니다. Python `input()` 대기를 3분에 강제로 끊는 타이머는 아닙니다. 목표 난이도와 실제 문제 난이도가 다를 수 있으므로 화면에는 실제 문제 난이도를 표시합니다.

In [ ]:
TOPIC = None
QUESTION_COUNT = 5
practice_session = LearningSession(questions, level=2, timeout_seconds=180, seed=SEED)
if INTERACTIVE:
    print(run_interactive(practice_session, count=QUESTION_COUNT, topic=TOPIC))
else:
    print('직접 풀기는 INTERACTIVE=True로 켤 수 있습니다.')

## 6. 복습 자료 저장 (선택)

`SAVE_REPORT=True`이면 **직접 푼 기록**을 `outputs` 폴더에 JSON과 HTML로 저장합니다. 합성 데모는 자동 저장하지 않습니다. 파일 이름에 고유 값을 붙여 기존 기록을 덮어쓰지 않습니다.

HTML에는 문제·내 답·정답·해설·피드백이 들어 있습니다. 브라우저에서 열어 **인쇄 → PDF로 저장**하면 한국어 학습지로 보관할 수 있습니다. 정답률과 풀이 시간은 해당 세션의 관찰값이며, 학습능력이나 집중력의 측정값이 아닙니다.

In [ ]:
if SAVE_REPORT and practice_session.history:
    report_paths = export_session(practice_session, PROJECT_DIR / 'outputs', label='practice')
    for path in report_paths:
        print(path)
elif SAVE_REPORT:
    print('저장할 실제 풀이 기록이 없습니다. 먼저 직접 문제를 풀어주세요.')
else:
    print('학습 기록 저장은 SAVE_REPORT=True로 켤 수 있습니다.')

## 7. GPT-2 파인튜닝과 문제 생성 (선택 실험)

외부 패키지·모델은 새로 설치하지 않았고, 다음 실험은 실행 검증하지 않았습니다. 사용할 정확한 패키지 버전·의존성·모델 파일을 먼저 검토해야 합니다. `docs/IMPLEMENTATION.md`의 외부 소스 검토 범위를 확인하세요. 자동 설치·Hub 다운로드 코드는 없습니다.

검토가 끝난 GPT-2 형식의 로컬 모델 디렉터리에는 토크나이저, `config.json`, safetensors 가중치가 있어야 합니다. `MODEL_DIR`을 지정하고 두 AI 실행 플래그를 켭니다. 이 플래그만 바꾸는 것으로 검토를 대신할 수 없습니다.

8/2 분할, 응답 부분만의 손실, 시드 고정, 검증 손실 기반 조기 종료를 사용합니다. 각 질문의 정답·풀이·피드백까지 학습합니다. 최대 길이를 넘으면 정답을 잘라 학습하지 않고 오류를 냅니다. 실험 결과가 학습 전 기준선보다 나빠지면 학습 전 가중치를 복구합니다. 10문항으로 생성 성능을 보장할 수 없습니다.

In [ ]:
if RUN_AI_EXPERIMENT:
    from ai_experiments import TrainingConfig, load_local_gpt2, fine_tune, generate_draft
    config = TrainingConfig(seed=SEED, epochs=3, learning_rate=5e-5, max_length=1024)
    model, tokenizer = load_local_gpt2(MODEL_DIR, reviewed_environment=REVIEWED_AI_ENV, seed=SEED)
    training_result = fine_tune(model, tokenizer, questions, config)
    print(json.dumps(training_result, ensure_ascii=False, indent=2))
else:
    print('GPT-2 학습 생략: 검토된 로컬 모델과 환경이 준비된 경우에만 실행합니다.')

### 생성 결과 검토

학습과 같은 프롬프트 형식에 주제·난이도를 넣고 새로 생성한 토큰만 읽습니다. JSON 형태가 잘못되면 오류로 보고하며, 잘못된 결과를 0점 답안처럼 취급하지 않습니다. JSON 형식이 맞아도 수학적 정답은 보장되지 않으므로 결과는 `needs_human_review` 초안입니다. 검토 전에는 문제은행이나 채점에 사용하지 않습니다.

In [ ]:
if RUN_AI_EXPERIMENT:
    draft = generate_draft(model, tokenizer, topic='집합', level=2,
                           context=material_text, max_new_tokens=256)
    print(json.dumps(draft, ensure_ascii=False, indent=2))

## 8. LSTM 학습 이력 실험 (선택)

기본 난이도 조절은 명시적인 규칙을 따릅니다. LSTM은 학습 이력을 분석하는 별도 실험으로 구성했습니다.

`ai_experiments.fit_history_lstm`은 한 학습자의 **실제 시간순 기록 40개 이상**을 받아, 과거 5회 기록의 문제 난이도·정답 여부·시간에서 다음 정답 여부를 예측하는 별도 실험입니다. 입력 형태는 `(batch, sequence, 3)`이고 PyTorch LSTM 뒤에 이진 분류 출력을 둡니다. 시간순 학습/검증 분할과 상수 예측 기준선 비교가 포함되어 있습니다. 40개는 최소 실행 조건일 뿐 충분한 데이터라는 뜻이 아닙니다.

이 데모에는 충분한 실제 이력이 없으므로 실행하지 않습니다. 아래 호출 예시는 실제 기록과 검토된 환경을 준비한 뒤 별도 셀에서 사용할 수 있습니다. 서로 다른 사람의 기록과 합성 데모를 섞지 마세요.

```python
from ai_experiments import fit_history_lstm
lstm_model, lstm_metrics = fit_history_lstm(
    real_attempts, reviewed_environment=REVIEWED_AI_ENV, window=5, epochs=20
)
```

LSTM 결과는 기본 난이도 정책에 자동 반영하지 않습니다. 별도 테스트 집합과 사용자 단위 검증이 필요합니다.

## 9. 남은 과제와 재현 범위

- 이 저장소에는 10문항 예제 데이터가 포함되며 학습 가중치는 포함되지 않습니다.
- 기본 풀이 흐름과 파일 저장은 테스트했고, 실제 모델 학습·생성 및 PDF 라이브러리 경로는 검증하지 않았습니다.
- 이 노트북의 합성 데모 기록은 실제 사용자의 결과나 모델 성능 평가가 아닙니다.
- 다양한 과목과 사용자에게 확장하려면 문제·정답 검토, 실제 이력 확보, 사용자별 평가가 필요합니다.

표준 라이브러리 테스트 실행: `python -m unittest discover -s tests -v`.

모델 API 참고: [Transformers 로컬 모델 로딩](https://huggingface.co/docs/transformers/v4.44.0/en/main_classes/model), [PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html).